# Music Annotation Project
## Notebook 90 - Post-A5 Residual Speck / Cosmetic Cleanup Audit

**Development diagnostic only. No retraining, refitting, recalibration or promotion authority.**

Purpose: test whether a narrow post-A5 residual cleanup layer can remove small annotation remnants without damaging genuine print. A1/A5 decisions remain frozen. Clean truth is used only for post-hoc scoring.

Arms:
1. A5 unchanged control.
2. Global 3x3 median diagnostic arm (expected to be unsafe; benchmark only).
3. Authority-constrained tiny-component cleanup sweeps over component area and print-distance veto.

Primary safety rule: no candidate may modify pixels outside frozen prepared authority.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib


In [ ]:
# ============================================================
# 1. Configuration and source resolution
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, gc, sys, platform
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N88_ROOT=ROOT/'a5_contextual_safe_cleanup'
SOURCE_N88_OVERRIDE=None
THRESHOLDS=[0.995,0.990,0.980,0.950]
AREA_LIMITS=[2,4,8,16,32]
DISTANCE_VETOES=[0,2,4,8]
DARK_THRESHOLD=200
PRINT_THRESHOLD=200
BOOTSTRAP_RESAMPLES=10000
SEED=42
REVIEW_RADIUS=3

REQUIRED=['tables/a5_primary_oof_predictions.parquet','tables/a5_threshold_sweep.csv','experiment_identity.json']
def complete_runs(root):
    if not root.exists(): return []
    return sorted([p for p in root.iterdir() if p.is_dir() and all((p/r).exists() for r in REQUIRED)],key=lambda p:p.stat().st_mtime)
if SOURCE_N88_OVERRIDE is not None:
    SOURCE_N88=Path(SOURCE_N88_OVERRIDE)
else:
    hits=complete_runs(N88_ROOT)
    if not hits:
        # Broad fallback for renamed Notebook 88 output root.
        hits=[]
        for p in ROOT.rglob('experiment_identity.json'):
            try:
                q=json.loads(p.read_text())
                if str(q.get('version','')).startswith('notebook88-a5') and all((p.parent/r).exists() for r in REQUIRED): hits.append(p.parent)
            except Exception: pass
        hits=sorted(hits,key=lambda p:p.stat().st_mtime)
    if not hits: raise RuntimeError('No complete Notebook 88 run found. Set SOURCE_N88_OVERRIDE.')
    SOURCE_N88=hits[-1]
identity=json.loads((SOURCE_N88/'experiment_identity.json').read_text())
SOURCE_N87=Path(identity['source_notebook87_run'])
RUN=ROOT/'a5_residual_cleanup_notebook90'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_n90_residual_cleanup')
TABLES=RUN/'tables'; AUDIT=RUN/'audit'; VIS=RUN/'visuals'; PAGES=RUN/'pages'
for p in [RUN,TABLES,AUDIT,VIS,PAGES]: p.mkdir(parents=True,exist_ok=False)

def require(c,m):
    if not c: raise RuntimeError(m)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def read_gray(path):
    path=Path(path)
    if path.suffix.lower()=='.npy':
        a=np.load(path,mmap_mode='r',allow_pickle=False); require(a.ndim==2,f'{path}: not 2D'); return np.array(a,dtype=np.uint8,copy=True)
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE); require(a is not None,f'Unable to read {path}'); return a
print('Notebook 88:',SOURCE_N88)
print('Notebook 87:',SOURCE_N87)
print('Output:',RUN)


In [ ]:
# ============================================================
# 2. Load durable Notebook 88 probabilities and reproduce policy
# ============================================================
PRED=SOURCE_N88/'tables'/'a5_primary_oof_predictions.parquet'
full=pd.read_parquet(PRED)
need={'page','source_group','y','x','label','p_annotation_a5'}
require(need<=set(full.columns),f'Missing columns: {sorted(need-set(full.columns))}')
full['page']=full.page.astype(str)
for c in ['y','x','label']: full[c]=pd.to_numeric(full[c],errors='raise').astype(int)
full['prob']=pd.to_numeric(full.p_annotation_a5,errors='raise').astype(float)
require(np.isfinite(full.prob).all(),'Non-finite A5 probabilities')
require(set(full.label.unique())<={0,1,2},'Expected labels 0=PRINT_KEEP, 1=ANNOTATION_CLEAN, 2=UNCERTAIN')

def run_filter(frame,selected,min_run=2):
    out=np.asarray(selected,bool).copy()
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy(); hit=idx[out[idx]]
        if len(hit)==0: continue
        h=int(g.y.max())+1; w=int(g.x.max())+1
        m=np.zeros((h,w),np.uint8); m[frame.loc[hit,'y'],frame.loc[hit,'x']]=1
        _,lab,stats,_=cv2.connectedComponentsWithStats(m,8)
        ids=lab[frame.loc[hit,'y'],frame.loc[hit,'x']]
        sizes=stats[ids,cv2.CC_STAT_AREA]
        out[hit[sizes<2]]=False
    return out
for t in THRESHOLDS:
    full[f'sel_{t:.3f}']=run_filter(full,(full.prob.to_numpy()>=t),2)
print('Rows:',len(full),'Pages:',full.page.nunique())


In [ ]:
# ============================================================
# 3. Resolve frozen page artefacts
# ============================================================
CACHE=SOURCE_N88/'observable_channel_cache'
if not CACHE.exists():
    candidates=list(SOURCE_N88.rglob('observable_channel_cache'))
    require(candidates,'Notebook 88 observable_channel_cache not found')
    CACHE=candidates[0]

def reg_clean(page):
    candidates=[SOURCE_N87/'truth_registration'/'registered_clean'/f'{page}_clean_registered.png',SOURCE_N88/'truth_registration'/'registered_clean'/f'{page}_clean_registered.png']
    for p in candidates:
        if p.exists(): return p
    raise RuntimeError(f'{page}: registered clean missing')

def validity(page):
    p=SOURCE_N87/'truth_registration'/'validity_masks'/f'{page}_validity.png'
    return p if p.exists() else None

records={}; missing=[]
for page in sorted(full.page.unique()):
    d=CACHE/page
    req={'observed':d/'observed.npy','a1':d/'a1.npy','paper':d/'paper.npy','authority':d/'authority.npy','ambiguous':d/'ambiguous.npy'}
    try: req['clean']=reg_clean(page)
    except Exception as e: missing.append((page,str(e))); continue
    absent=[k for k,p in req.items() if not p.exists()]
    if absent: missing.append((page,'missing '+','.join(absent))); continue
    records[page]=req
pd.DataFrame(missing,columns=['page','reason']).to_csv(AUDIT/'missing_inputs.csv',index=False)
require(not missing,f'{len(missing)} pages have missing inputs')
print('Resolved pages:',len(records))


In [ ]:
# ============================================================
# 4. Residual-component inventory after A5 at every threshold
# ============================================================
rows=[]
for page,g in full.groupby('page',sort=False):
    r=records[page]
    obs=read_gray(r['observed']); a1=read_gray(r['a1']); paper=read_gray(r['paper']); authority=read_gray(r['authority'])>0; clean=read_gray(r['clean'])
    require(obs.shape==a1.shape==paper.shape==authority.shape==clean.shape,f'{page}: shape mismatch')
    truth_print=cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
    # distance from registered clean printed ink, target coordinates
    dist=cv2.distanceTransform((~truth_print).astype(np.uint8),cv2.DIST_L2,5)
    ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    for t in THRESHOLDS:
        a5=a1.copy(); sel=g[f'sel_{t:.3f}'].to_numpy(bool); a5[ys[sel],xs[sel]]=paper[ys[sel],xs[sel]]
        residual=(a5<DARK_THRESHOLD)&authority
        n,lab,stats,_=cv2.connectedComponentsWithStats(residual.astype(np.uint8),8)
        for cid in range(1,n):
            x,y,w,h,area=stats[cid]
            m=(lab==cid)
            # truth mix is evaluated only at component pixels
            pr=int((m&truth_print).sum()); ann=int((m&~truth_print).sum())
            rows.append({'page':page,'source_group':g.source_group.iloc[0],'threshold':t,'component_id':cid,'x':int(x),'y':int(y),'w':int(w),'h':int(h),'area':int(area),'truth_print_pixels':pr,'truth_nonprint_pixels':ann,'min_distance_to_print':float(dist[m].min()) if m.any() else np.nan,'mean_distance_to_print':float(dist[m].mean()) if m.any() else np.nan})
        del a5,residual,lab,stats
    del obs,a1,paper,authority,clean,truth_print,dist
    gc.collect()
components=pd.DataFrame(rows)
components.to_parquet(TABLES/'residual_component_inventory.parquet',index=False)
print('Residual components:',len(components))


In [ ]:
# ============================================================
# 5. Sweep tiny-component cleanup: area x distance-to-print veto
# ============================================================
summary=[]; page_rows=[]
for t in THRESHOLDS:
    q=components[components.threshold==t]
    for area_limit in AREA_LIMITS:
        for distance_veto in DISTANCE_VETOES:
            chosen=q[(q.area<=area_limit)&(q.min_distance_to_print>=distance_veto)]
            ann=int(chosen.truth_nonprint_pixels.sum()); pr=int(chosen.truth_print_pixels.sum())
            cats=int(((chosen.truth_print_pixels>=20)&(chosen.truth_print_pixels>chosen.truth_nonprint_pixels)).sum())
            summary.append({'threshold':t,'area_limit':area_limit,'distance_veto':distance_veto,'components_selected':len(chosen),'annotation_like_pixels_removed':ann,'print_pixels_at_risk':pr,'print_per_annotation':pr/max(ann,1),'utility':ann-5*pr-500*cats,'catastrophic_components':cats,'pages':chosen.page.nunique(),'source_groups':chosen.source_group.nunique()})
            z=chosen.groupby(['page','source_group'],as_index=False).agg(components_selected=('component_id','size'),annotation_like_pixels_removed=('truth_nonprint_pixels','sum'),print_pixels_at_risk=('truth_print_pixels','sum')) if len(chosen) else pd.DataFrame(columns=['page','source_group','components_selected','annotation_like_pixels_removed','print_pixels_at_risk'])
            if len(z):
                z['threshold']=t;z['area_limit']=area_limit;z['distance_veto']=distance_veto;page_rows.append(z)
sweep=pd.DataFrame(summary)
sweep.to_csv(TABLES/'tiny_component_sweep.csv',index=False)
(pd.concat(page_rows,ignore_index=True) if page_rows else pd.DataFrame()).to_csv(TABLES/'tiny_component_sweep_by_page.csv',index=False)
print(sweep.sort_values(['print_pixels_at_risk','annotation_like_pixels_removed'],ascending=[True,False]).head(30).to_string(index=False))


In [ ]:
# ============================================================
# 6. Global 3x3 median benchmark (diagnostic only)
# ============================================================
median_rows=[]
for page,g in full.groupby('page',sort=False):
    r=records[page]; obs=read_gray(r['observed']); a1=read_gray(r['a1']); paper=read_gray(r['paper']); authority=read_gray(r['authority'])>0; clean=read_gray(r['clean'])
    truth_print=cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
    ys=g.y.to_numpy(int);xs=g.x.to_numpy(int)
    for t in THRESHOLDS:
        a5=a1.copy(); sel=g[f'sel_{t:.3f}'].to_numpy(bool);a5[ys[sel],xs[sel]]=paper[ys[sel],xs[sel]]
        med=cv2.medianBlur(a5,3)
        changed=med!=a5
        median_rows.append({'page':page,'threshold':t,'changed_pixels':int(changed.sum()),'changed_outside_authority':int((changed&~authority).sum()),'changed_truth_print':int((changed&truth_print).sum())})
        del a5,med,changed
    del obs,a1,paper,authority,clean,truth_print;gc.collect()
median=pd.DataFrame(median_rows);median.to_csv(TABLES/'global_median3_diagnostic.csv',index=False)
print(median.groupby('threshold',as_index=False)[['changed_pixels','changed_outside_authority','changed_truth_print']].sum().to_string(index=False))


In [ ]:
# ============================================================
# 7. Select diagnostic policies and build high-resolution examples
# ============================================================
# Strictly diagnostic: choose best zero-print-risk rows first; otherwise lowest risk.
rank=sweep.sort_values(['print_pixels_at_risk','catastrophic_components','annotation_like_pixels_removed'],ascending=[True,True,False]).reset_index(drop=True)
rank.to_csv(TABLES/'ranked_tiny_component_policies.csv',index=False)
chosen=rank.iloc[0].to_dict()
print('Top development diagnostic policy:',chosen)

# Review queues: largest apparent benefit and any print-risk components under that policy.
t=float(chosen['threshold']); area=int(chosen['area_limit']); dv=float(chosen['distance_veto'])
eligible_components=components[(components.threshold==t)&(components.area<=area)&(components.min_distance_to_print>=dv)].copy()
benefit=eligible_components.sort_values(['truth_nonprint_pixels','truth_print_pixels'],ascending=[False,True]).head(50)
risk=eligible_components[eligible_components.truth_print_pixels>0].sort_values(['truth_print_pixels','truth_nonprint_pixels'],ascending=False).head(50)
benefit.to_csv(TABLES/'review_queue_benefit.csv',index=False);risk.to_csv(TABLES/'review_queue_risk.csv',index=False)

def overlay_mask(gray,mask,bgr,alpha=.65):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); colour=base.copy();colour[mask]=bgr;out=base.copy();out[mask]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[mask];return out

def save_examples(frame,prefix,limit=30):
    for i,row in enumerate(frame.head(limit).itertuples(),1):
        page=str(row.page); r=records[page]; obs=read_gray(r['observed']);clean=read_gray(r['clean']);a1=read_gray(r['a1']);paper=read_gray(r['paper']);authority=read_gray(r['authority'])>0
        g=full[full.page==page];ys=g.y.to_numpy(int);xs=g.x.to_numpy(int);a5=a1.copy();sel=g[f'sel_{t:.3f}'].to_numpy(bool);a5[ys[sel],xs[sel]]=paper[ys[sel],xs[sel]]
        residual=(a5<DARK_THRESHOLD)&authority;_,lab,_,_=cv2.connectedComponentsWithStats(residual.astype(np.uint8),8);m=(lab==int(row.component_id))
        pad=100;x0=max(0,int(row.x)-pad);y0=max(0,int(row.y)-pad);x1=min(obs.shape[1],int(row.x+row.w)+pad);y1=min(obs.shape[0],int(row.y+row.h)+pad)
        proposed=a5.copy();proposed[m]=paper[m]
        panels=[obs[y0:y1,x0:x1],clean[y0:y1,x0:x1],a1[y0:y1,x0:x1],a5[y0:y1,x0:x1],overlay_mask(obs,m,(0,255,255))[y0:y1,x0:x1],proposed[y0:y1,x0:x1]]
        titles=['Observed','Registered clean','A1',f'A5 {t:.3f}','Residual component','A5 + speck cleanup']
        fig,axes=plt.subplots(2,3,figsize=(15,10));
        for ax,im,title in zip(axes.flat,panels):
            if im.ndim==3: im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB);ax.imshow(im)
            else: ax.imshow(im,cmap='gray',vmin=0,vmax=255)
            ax.set_title(title);ax.axis('off')
        fig.suptitle(f'{page} component {int(row.component_id)} | area={int(row.area)} | truth print={int(row.truth_print_pixels)} | nonprint={int(row.truth_nonprint_pixels)} | min dist={row.min_distance_to_print:.2f}')
        fig.tight_layout();fig.savefig(VIS/f'{prefix}_{i:02d}_{page}_{int(row.component_id):05d}.png',dpi=150);plt.close(fig)
        del obs,clean,a1,paper,authority,a5,residual,lab,m,proposed,panels;gc.collect()
save_examples(benefit,'benefit');save_examples(risk,'risk')
print('Visual examples:',VIS)


In [ ]:
# ============================================================
# 8. Page-clustered bootstrap for the selected diagnostic rule
# ============================================================
chosen_components=eligible_components.copy()
per_page=chosen_components.groupby(['page','source_group'],as_index=False).agg(annotation_like=('truth_nonprint_pixels','sum'),print_risk=('truth_print_pixels','sum'),components=('component_id','size'))
all_pages=sorted(full.page.unique()); base=pd.DataFrame({'page':all_pages}).merge(per_page,on='page',how='left').fillna({'annotation_like':0,'print_risk':0,'components':0,'source_group':'NONE'})
rng=np.random.default_rng(SEED);boot=np.empty((BOOTSTRAP_RESAMPLES,2),float)
for i in range(BOOTSTRAP_RESAMPLES):
    q=base.iloc[rng.integers(0,len(base),len(base))];a=q.annotation_like.sum();p=q.print_risk.sum();boot[i]=[a-5*p,p/max(a,1)]
bootstrap={'utility_95_low':float(np.quantile(boot[:,0],.025)),'utility_95_high':float(np.quantile(boot[:,0],.975)),'print_per_annotation_95_high':float(np.quantile(boot[:,1],.975))}
(TABLES/'selected_policy_bootstrap.json').write_text(json.dumps(bootstrap,indent=2));print(bootstrap)


In [ ]:
# ============================================================
# 9. Final research verdict and identity
# ============================================================
selected_summary={**chosen,**bootstrap}
median_primary=median[median.threshold==t][['changed_pixels','changed_outside_authority','changed_truth_print']].sum().to_dict()
verdict={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'notebook':'90_A5_Post_Correction_Residual_Speck_Cleanup_Audit',
 'development_only':True,'retrained':False,'refitted':False,'recalibrated':False,'production_authorisation':False,
 'source_notebook88':str(SOURCE_N88),'source_notebook87':str(SOURCE_N87),
 'question':'Can a narrow post-A5 cleanup remove small residual annotation components without genuine-print loss?',
 'selected_diagnostic_policy':selected_summary,
 'global_median3_diagnostic':median_primary,
 'decision':('PROMISING_FOR_SEPARATE_PREREGISTERED_CHALLENGER' if int(chosen['print_pixels_at_risk'])==0 and int(chosen['annotation_like_pixels_removed'])>0 and bootstrap['utility_95_low']>0 else 'NO_SAFE_RESIDUAL_RULE_ESTABLISHED'),
 'note':'No policy is promoted by this notebook. Any candidate requires a separately pre-registered challenger and untouched validation.'
}
(RUN/'notebook90_verdict.json').write_text(json.dumps(verdict,indent=2,default=str))
identity={'version':'notebook90-a5-post-correction-residual-audit','source_n88_identity_sha256':sha256_file(SOURCE_N88/'experiment_identity.json'),'source_predictions_sha256':sha256_file(PRED),'thresholds':THRESHOLDS,'area_limits':AREA_LIMITS,'distance_vetoes':DISTANCE_VETOES,'dark_threshold':DARK_THRESHOLD,'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__}}
(RUN/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))
lines=['NOTEBOOK 90 - POST-A5 RESIDUAL CLEANUP AUDIT','',f'Source N88: {SOURCE_N88}',f'Rows: {len(full):,}',f'Pages: {full.page.nunique()}',f'Top diagnostic rule: A5 {t:.3f}, component area <= {area}, distance-to-print >= {dv}',f'Annotation-like residual pixels: {int(chosen["annotation_like_pixels_removed"]):,}',f'Print pixels at risk: {int(chosen["print_pixels_at_risk"]):,}',f'Clustered utility 95%: {bootstrap["utility_95_low"]:.1f} to {bootstrap["utility_95_high"]:.1f}',f'Global median 3x3 changed truth-print pixels at comparable A5 threshold: {int(median_primary["changed_truth_print"]):,}','',f'Decision: {verdict["decision"]}','Development diagnostic only. No automatic promotion.']
(RUN/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines));print('\nOutput:',RUN)
